# Usar Qwen 2.5 3B para clasificar noticias usando Colab gratuito

In [11]:
# ANTES DE EJECUTARLO:
# Vaya a Runtime -> Change Runtime Type -> y seleccione "T4 GPU"
#
!pip install -q -U transformers bitsandbytes accelerate

In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Model ID para Qwen 2.5 1.5B Instruct
model_id = "Qwen/Qwen2.5-3B-Instruct"

# 1. Configurar al modelo cuantizado a 4-bits
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Cargar el mismo tokenizador utilizado para entrenar el modelo
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 3. Cargar el modelo cuantizado a 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("\n✅ Model cargado a T4 GPU!")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


✅ Model cargado a T4 GPU!


In [13]:
def ask_qwen(user_prompt, system_prompt="You are a helpful assistant."):
    """
    Los prompts son rodeados por tokens especiales antes de enviarlos al modelo. El modelo fue entrenado
    especificamente para reconocer estos tokens y para prestarle mas atención al system prompt.
    :param system_promt: Las instrucciones en el system prompt son "leyes".
    :param user_prompt: texto en el user prompt es interpretado como "pedido conteniendo informacion"
     a ser procesado bajo las reglas del system prompt.
    """
    # Formatear el prompt usando el template de chat del modelo, que separa system y user prompt.
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generar la respuesta
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    # Extraer de la respuesta solo los tokens de respuesta, ignorando lo enviado
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response


In [ ]:
# Este es un ejemplo (pobre) de few-shot learning enseñarle al modelo a clasificar noticias.
# Muy probablemente necesite agregar mas ejemplos, tiene que cambiar las categorias, y
# puede cambiar todo lo que quiera para mejorar el resultado.
#
system_prompt="You are an expert classifying news into categories. Always reply with a single category."
prompt_text = """
This is a set of news, each with a category. Each news starts with <EXAMPLE> and ends with </EXAMPLE>, each news belongs to a single category that stars with <CATEGORY> and ends with </CATEGORY>:

<EXAMPLE>
<INPUT>
Se trabó el acuerdo entre Karina Milei y los Macri para compartir las listas en la Capital.
La negociación entre LLA y Pro se interrumpió porque no logran consensuar el reparto de lugares en la nómina de diputados nacionales; malestar y reproches cruzados.
La negociación entre Karina Milei y los Macri para que La Libertad Avanza (LLA) y Pro conformen un frente electoral en la Capital para competir en las legislativas nacionales del 26 de octubre se trabó durante las últimas horas.
Después de que se produjeran los primeros intercambios para llegar a un entendimiento en la ciudad de Buenos Aires, libertarios y macristas no logran acercar posiciones para oficializar el armado de un frente conjunto el próximo jueves, cuando vencerá el plazo legal para anotar las alianzas.
</INPUT>
<CATEGORY>politica</CATEGORY>
</EXAMPLE>

<EXAMPLE>
<INPUT>
El sueño de un Amazon local y una pelea sin tregua contra el cisne negro
El Gobierno confía en que el ecosistema comercial que armó blinde parcialmente los precios contra la suba del tipo de cambio; el FMI
empeoró la expectativa de déficit de cuenta corriente, anticipo de que el dólar seguirá siendo un problema.
</INPUT>
<CATEGORY>economía</CATEGORY>
</EXAMPLE>

Complete the following text with **ONLY** one of these categories: politica, economía.
<EXAMPLE>
<INPUT>
La suba del dólar tendrá bajo impacto en la inflación de los próximos meses según los analistas que releva el BCRA
</INPUT>
"""
print(ask_qwen(prompt_text, system_prompt))

economía


In [ ]:
# Este es un ejemplo (pobre) de few-shot learning enseñarle al modelo a clasificar noticias.
# Muy probablemente necesite agregar mas ejemplos, tiene que cambiar las categorias, y
# puede cambiar todo lo que quiera para mejorar el resultado.
#
system_prompt="You are an expert classifying news into categories. Always reply with a single category."
prompt_text = """
This is a set of news, each with a category. Each news starts with <EXAMPLE> and ends with </EXAMPLE>, each news belongs to a single category that stars with <CATEGORY> and ends with </CATEGORY>:

<EXAMPLE>
<INPUT>
Se trabó el acuerdo entre Karina Milei y los Macri para compartir las listas en la Capital.
La negociación entre LLA y Pro se interrumpió porque no logran consensuar el reparto de lugares en la nómina de diputados nacionales; malestar y reproches cruzados.
La negociación entre Karina Milei y los Macri para que La Libertad Avanza (LLA) y Pro conformen un frente electoral en la Capital para competir en las legislativas nacionales del 26 de octubre se trabó durante las últimas horas.
Después de que se produjeran los primeros intercambios para llegar a un entendimiento en la ciudad de Buenos Aires, libertarios y macristas no logran acercar posiciones para oficializar el armado de un frente conjunto el próximo jueves, cuando vencerá el plazo legal para anotar las alianzas.
</INPUT>
<CATEGORY>politica</CATEGORY>
</EXAMPLE>

<EXAMPLE>
<INPUT>
El sueño de un Amazon local y una pelea sin tregua contra el cisne negro
El Gobierno confía en que el ecosistema comercial que armó blinde parcialmente los precios contra la suba del tipo de cambio; el FMI
empeoró la expectativa de déficit de cuenta corriente, anticipo de que el dólar seguirá siendo un problema.
</INPUT>
<CATEGORY>economía</CATEGORY>
</EXAMPLE>

Complete the following text with **ONLY** one of these categories: politica, economía.
<EXAMPLE>
<INPUT>
La suba del dólar tendrá bajo impacto en la inflación de los próximos meses según los analistas que releva el BCRA
</INPUT>
"""
print(ask_qwen(prompt_text, system_prompt))

economía


In [ ]:
# Este es un ejemplo (pobre) de few-shot learning enseñarle al modelo a clasificar noticias.
# Muy probablemente necesite agregar mas ejemplos, tiene que cambiar las categorias, y
# puede cambiar todo lo que quiera para mejorar el resultado.
#
system_prompt="You are an expert classifying news into categories. Always reply with a single category."
prompt_text = """
This is a set of news, each with a category. Each news starts with <EXAMPLE> and ends with </EXAMPLE>, each news belongs to a single category that stars with <CATEGORY> and ends with </CATEGORY>:

<EXAMPLE>
<INPUT>
Se trabó el acuerdo entre Karina Milei y los Macri para compartir las listas en la Capital.
La negociación entre LLA y Pro se interrumpió porque no logran consensuar el reparto de lugares en la nómina de diputados nacionales; malestar y reproches cruzados.
La negociación entre Karina Milei y los Macri para que La Libertad Avanza (LLA) y Pro conformen un frente electoral en la Capital para competir en las legislativas nacionales del 26 de octubre se trabó durante las últimas horas.
Después de que se produjeran los primeros intercambios para llegar a un entendimiento en la ciudad de Buenos Aires, libertarios y macristas no logran acercar posiciones para oficializar el armado de un frente conjunto el próximo jueves, cuando vencerá el plazo legal para anotar las alianzas.
</INPUT>
<CATEGORY>politica</CATEGORY>
</EXAMPLE>

<EXAMPLE>
<INPUT>
El sueño de un Amazon local y una pelea sin tregua contra el cisne negro
El Gobierno confía en que el ecosistema comercial que armó blinde parcialmente los precios contra la suba del tipo de cambio; el FMI
empeoró la expectativa de déficit de cuenta corriente, anticipo de que el dólar seguirá siendo un problema.
</INPUT>
<CATEGORY>economía</CATEGORY>
</EXAMPLE>

Complete the following text with **ONLY** one of these categories: politica, economía.
<EXAMPLE>
<INPUT>
La suba del dólar tendrá bajo impacto en la inflación de los próximos meses según los analistas que releva el BCRA
</INPUT>
"""
print(ask_qwen(prompt_text, system_prompt))

economía


In [ ]:
# Este es un ejemplo (pobre) de few-shot learning enseñarle al modelo a clasificar noticias.
# Muy probablemente necesite agregar mas ejemplos, tiene que cambiar las categorias, y
# puede cambiar todo lo que quiera para mejorar el resultado.
#
system_prompt="You are an expert classifying news into categories. Always reply with a single category."
prompt_text = """
This is a set of news, each with a category. Each news starts with <EXAMPLE> and ends with </EXAMPLE>, each news belongs to a single category that stars with <CATEGORY> and ends with </CATEGORY>:

<EXAMPLE>
<INPUT>
Se trabó el acuerdo entre Karina Milei y los Macri para compartir las listas en la Capital.
La negociación entre LLA y Pro se interrumpió porque no logran consensuar el reparto de lugares en la nómina de diputados nacionales; malestar y reproches cruzados.
La negociación entre Karina Milei y los Macri para que La Libertad Avanza (LLA) y Pro conformen un frente electoral en la Capital para competir en las legislativas nacionales del 26 de octubre se trabó durante las últimas horas.
Después de que se produjeran los primeros intercambios para llegar a un entendimiento en la ciudad de Buenos Aires, libertarios y macristas no logran acercar posiciones para oficializar el armado de un frente conjunto el próximo jueves, cuando vencerá el plazo legal para anotar las alianzas.
</INPUT>
<CATEGORY>politica</CATEGORY>
</EXAMPLE>

<EXAMPLE>
<INPUT>
El sueño de un Amazon local y una pelea sin tregua contra el cisne negro
El Gobierno confía en que el ecosistema comercial que armó blinde parcialmente los precios contra la suba del tipo de cambio; el FMI
empeoró la expectativa de déficit de cuenta corriente, anticipo de que el dólar seguirá siendo un problema.
</INPUT>
<CATEGORY>economía</CATEGORY>
</EXAMPLE>

Complete the following text with **ONLY** one of these categories: politica, economía.
<EXAMPLE>
<INPUT>
La suba del dólar tendrá bajo impacto en la inflación de los próximos meses según los analistas que releva el BCRA
</INPUT>
"""
print(ask_qwen(prompt_text, system_prompt))

economía


In [ ]:
# Este es un ejemplo extra de uso de Qwen: Use el siguiente codigo para usar el LLM como un chatbot.
#
# print("Escriba 'chau' para terminar el chat.")
# while True:
#     user_input = input("Ud: ")
#     if user_input.lower() == 'chau':
#         break
#
#     output = ask_qwen(user_input)
#     print(f"\nQwen: {output}\n" + "-"*50)